<a href="https://colab.research.google.com/github/htf100/transfomer/blob/main/docs_nnx/chinese_aomen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r "/content/drive/MyDrive/chinese_common" "/content/"


In [3]:
!pwd

/content


In [4]:
%cd /content/chinese_common


/content/chinese_common


In [6]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 64.0 MB/s eta 0:00:00


In [5]:
"""训练：python dict_simple.py；预测：python dict_simple.py "服务很好" "体验很差"。"""

import csv
import sys
from pathlib import Path

import lightning.pytorch as pl
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

# 常用配置集中在这里；数据和模型路径相对于本文件。
ROOT = Path(__file__).resolve().parent
CHECKPOINT = ROOT / "review_simple.ckpt"
MAX_LENGTH, HIDDEN_SIZE = 32, 32
BATCH_SIZE, EPOCHS, PATIENCE = 32, 20, 3
LR, SEED = 1e-3, 42
USE_GPU = torch.cuda.is_available()  # 有 CUDA 就使用 GPU；改为 False 可强制 CPU。
GPU_ID = 0                         # 使用第几张可见的 GPU，从 0 开始。
UNK, PAD = 0, 1


def read_reviews(path):
    """固定读取带 label,text 表头的 CSV；0 是差评，1 是好评。"""
    with open(path, encoding="utf-8-sig", newline="") as file:
        samples = [(row["text"].strip(), int(row["label"])) for row in csv.DictReader(file)]
    if not samples or any(not text or label not in (0, 1) for text, label in samples):
        raise ValueError(f"{path}：需要非空评论和 0/1 标签")
    return samples


def encode(text, vocab, max_length):
    text = text.strip()
    if not text:
        raise ValueError("评论不能为空")
    tokens = [vocab.get(char, UNK) for char in text[:max_length]]
    return tokens + [PAD] * (max_length - len(tokens))


def make_loader(samples, vocab, max_length, shuffle=False):
    tokens = torch.tensor([encode(text, vocab, max_length) for text, _ in samples])
    labels = torch.tensor([label for _, label in samples])
    return DataLoader(
        TensorDataset(tokens, labels), batch_size=BATCH_SIZE,
        shuffle=shuffle, pin_memory=USE_GPU,
    )


class ReviewModel(pl.LightningModule):
    def __init__(self, vocab, max_length=64, hidden_size=128, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()  # 词表和结构参数随模型保存，预测时自动恢复。
        self.embedding = nn.Embedding(len(vocab), hidden_size, padding_idx=PAD)
        self.position_embedding = nn.Embedding(max_length, hidden_size)
        self.linear_q = nn.Linear(hidden_size, hidden_size)
        self.linear_k = nn.Linear(hidden_size, hidden_size)
        self.linear_v = nn.Linear(hidden_size, hidden_size)
        self.layer_norm_0 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 2), nn.ReLU(),
            nn.Linear(hidden_size * 2, hidden_size),
        )
        self.layer_norm_1 = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, tokens):
        mask = tokens != PAD                              # [B, L]，True 是真实字符。
        positions = torch.arange(tokens.size(1), device=tokens.device)
        x = self.embedding(tokens) + self.position_embedding(positions)
        q = self.linear_q(x).unsqueeze(1)                  # [B, 1, L, H]，单头注意力。
        k = self.linear_k(x).unsqueeze(1)
        v = self.linear_v(x).unsqueeze(1)
        attended = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask[:, None, None, :], dropout_p=0.0,
        ).squeeze(1)
        x = self.layer_norm_0(x + attended)
        x = self.layer_norm_1(x + self.ffn(x))
        valid = mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * valid).sum(1) / valid.sum(1)          # 平均池化，排除 PAD。
        return self.classifier(pooled)                    # [B, 2]，直接用于交叉熵。

    def shared_step(self, batch, stage):
        tokens, labels = batch
        logits = self(tokens)
        loss = F.cross_entropy(logits, labels)
        accuracy = (logits.argmax(1) == labels).float().mean()
        self.log_dict(
            {f"{stage}_loss": loss, f"{stage}_acc": accuracy},
            on_step=False, on_epoch=True, prog_bar=True, batch_size=len(labels),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self.shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self.shared_step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)


@torch.inference_mode()
def predict(model, texts):
    model.eval()
    for text in texts:
        tokens = encode(text, model.hparams.vocab, model.hparams.max_length)
        inputs = torch.tensor([tokens], device=model.device)
        probabilities = model(inputs).softmax(dim=-1)[0]
        label = probabilities.argmax().item()
        print(f"{text} → {['差评', '好评'][label]}（模型概率 {probabilities[label].item():.1%}）")
        if len(text.strip()) > model.hparams.max_length:
            print(f"  仅分析前 {model.hparams.max_length} 个字符")


def train():
    pl.seed_everything(SEED, workers=True)
    training = read_reviews(ROOT / "train.csv")
    validation = read_reviews(ROOT / "valid.csv")
    vocab = {"[UNK]": UNK, "[PAD]": PAD}
    for text, _ in training:  # 只用训练集建词表，验证和测试中的新字符映射为 UNK。
        for char in text:
            vocab.setdefault(char, len(vocab))
    train_loader = make_loader(training, vocab, MAX_LENGTH, shuffle=True)
    val_loader = make_loader(validation, vocab, MAX_LENGTH)
    print(f"训练 {len(training)} 条 | 验证 {len(validation)} 条 | 词表 {len(vocab)}")
    print(f"训练设备：{f'cuda:{GPU_ID}' if USE_GPU else 'cpu'}")

    checkpoint = ModelCheckpoint(
        dirpath=CHECKPOINT.parent, filename=CHECKPOINT.stem,
        monitor="val_loss", mode="min", save_top_k=1, enable_version_counter=False,
    )
    # GPU 训练：Lightning 自动把模型和每个 batch 搬到指定显卡，并管理反向传播。
    trainer = pl.Trainer(
        accelerator="gpu" if USE_GPU else "cpu",
        devices=[GPU_ID] if USE_GPU else 1,
        max_epochs=EPOCHS, logger=False, num_sanity_val_steps=0,
        callbacks=[checkpoint, EarlyStopping(monitor="val_loss", patience=PATIENCE)],
    )
    model = ReviewModel(vocab, MAX_LENGTH, HIDDEN_SIZE, LR)
    trainer.fit(model, train_loader, val_loader)
    best = ReviewModel.load_from_checkpoint(
        checkpoint.best_model_path, map_location="cpu", weights_only=True,
    )
    print(f"最佳模型：{checkpoint.best_model_path}")
    # 训练完成后才评估测试集；测试集不参与模型选择。
    testing = read_reviews(ROOT / "test.csv")
    trainer.test(best, dataloaders=make_loader(testing, vocab, MAX_LENGTH))
    return best


if __name__ == "__main__":
    if len(sys.argv) > 1:
        model = ReviewModel.load_from_checkpoint(CHECKPOINT, map_location="cpu", weights_only=True)
        model.to(f"cuda:{GPU_ID}" if USE_GPU else "cpu")
        predict(model, sys.argv[1:])
    else:
        model = train()
        predict(model, ["服务很好，下次还来", "等了很久，体验很差"])



Traceback (most recent call last):
  File "/content/chinese_common/dict_simple.py", line 7, in <module>
    import lightning.pytorch as pl
ModuleNotFoundError: No module named 'lightning'
